# The InTweetionists
## Deep Learning pipeline for tweet classification

### Useful imports

In [1]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.metrics import make_scorer, accuracy_score
import numpy as np
import xgboost as xgb
import json
import pandas as pd
from pandas import json_normalize
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from textblob import TextBlob
import re
from sklearn.ensemble import RandomForestClassifier
from gensim.models import Word2Vec
from gensim.models import Word2Vec

from nltk.corpus import stopwords

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.nn.functional as F

### Loading of the data contained in the JSON

In [2]:
# Load the training data from a JSON Lines file (one JSON object per line)
train_data = pd.read_json('train.jsonl', lines=True)
# The tweet data is nested. json_normalize flattens the nested JSON into columns.
train_data = json_normalize(train_data.to_dict(orient='records'))

# Load the Kaggle test data (which we will make predictions on)
kaggle_data = pd.read_json('kaggle_test.jsonl', lines=True)
# Also normalize the Kaggle data
kaggle_data = json_normalize(kaggle_data.to_dict(orient='records'))


# Separate features from the target variable for the training set
X_train = train_data.drop('label', axis=1)
y_train = train_data['label']

X_kaggle = kaggle_data

print("Data successfully loaded ✓")

Data successfully loaded ✓


### Definition of two main functions for feature engineering

In [3]:
def create_advanced_features(df_input):
    """Create new columns in the dataframe given as input to account for 
    new features that could be useful for the classification.
    
    Args:
        df_input (pandas.DataFrame): the dataframe given as input. It is 
        not impacted since this function works on a copy.
    
    Returns:
        df (pandas.DataFrame): a copy of df_input with the additional 
        columns. 
    """
    df = df_input.copy()
    
    # Definition of fallback series (robustness against missing columns)
    default_int_series = pd.Series(0, index=df.index)
    default_bool_series = pd.Series(False, index=df.index)
    
    # --- Initialization of numerical columns ---
    df['user.followers_count'] = df.get('user.followers_count', default_int_series).fillna(0)
    df['user.friends_count'] = df.get('user.friends_count', default_int_series).fillna(0)
    df['user.listed_count'] = df.get('user.listed_count', default_int_series).fillna(0)
    df['user.favourites_count'] = df.get('user.favourites_count', default_int_series).fillna(0)
    df['user.statuses_count'] = df.get('user.statuses_count', default_int_series).fillna(0)
    df['retweet_count'] = df.get('retweet_count', default_int_series).fillna(0)
    df['favorite_count'] = df.get('favorite_count', default_int_series).fillna(0)
    df['quote_count'] = df.get('quote_count', default_int_series).fillna(0) # Nouveau : Quote Count
    df['reply_count'] = df.get('reply_count', default_int_series).fillna(0) # Nouveau : Reply Count
    
    # --- A. HANDLING OF DATES (Account's age) ---
    df['user_created_at_dt'] = pd.to_datetime(df.get('user.created_at'), errors='coerce')
    ref_date = pd.to_datetime('now', utc=True)
    df['account_age_days'] = (ref_date - df['user_created_at_dt']).dt.days
    df['account_age_days'] = df['account_age_days'].fillna(0)
    
    # Extraction of the tweet's advanced time indicators
    df['created_at_dt'] = pd.to_datetime(df.get('created_at'), errors='coerce')
    df['tweet_hour'] = df['created_at_dt'].dt.hour.fillna(-1)
    df['tweet_is_weekend'] = df['created_at_dt'].dt.dayofweek.isin([5, 6]).fillna(False).astype(int)

    # --- B. PROFILE'S QUALITY & STATUS (Booleans) ---
    df['is_default_profile'] = df.get('user.default_profile', default_bool_series).fillna(False).astype(int)
    df['is_default_image'] = df.get('user.default_profile_image', default_bool_series).fillna(False).astype(int)
    df['is_verified'] = df.get('user.verified', default_bool_series).fillna(False).astype(int)
    
    # New : Is it a protected or private account ? (Typical of a follower or personal account)
    df['is_protected'] = df.get('user.protected', default_bool_series).fillna(False).astype(int)
    
    # New : Does the progile have a URL ?
    df['has_url'] = df.get('user.url', pd.Series(False, index=df.index)).notna().astype(int)

    # --- C. TWEET'S CONTENT (Entities Counting) ---
    def count_entities(x):
        if isinstance(x, list) or (isinstance(x, pd.Series) and x.dtype == object): return len(x)
        return 0

    df['num_urls'] = df.get('entities.urls', default_int_series).apply(count_entities)
    df['num_hashtags'] = df.get('entities.hashtags', default_int_series).apply(count_entities)
    df['num_mentions'] = df.get('entities.user_mentions', default_int_series).apply(count_entities)
    df['has_media'] = df.get('extended_entities.media', default_bool_series).notna().astype(int)

    # --- D. POWERFUL AND BEHAVIORAL RATIOS ---
    followers = df['user.followers_count']
    friends = df['user.friends_count']
    listed = df['user.listed_count']
    statuses = df['user.statuses_count']
    
    # 1. Followers / Friends ratio (Fame ratio)
    df['ratio_followers_friends'] = followers / (friends + 1)
    df['ratio_listed_followers'] = listed / (followers + 1)
    
    # 2. Friendship reciprocity rate (New - Indicates an influence balance or imbalance)
    df['reciprocity_score'] = (friends - followers) / (friends + followers + 1)

    # 3. Activity (Tweets per day since the creation of the account)
    df['tweets_per_day'] = statuses / (df['account_age_days'] + 1)
    
    # 4. Mention/Tweet ratio (New - Interaction vs. diffusion ratio)
    # The higher this ratio, the more the user personally  (follower).
    df['ratio_mention_status'] = df['num_mentions'] / (statuses + 1)
    
    # 5. Engagement (Engagement rate per Tweet)
    total_engagement = df['retweet_count'] + df['favorite_count'] + df['quote_count'] + df['reply_count']
    df['total_tweet_engagement'] = total_engagement / (followers + 1)

    # --- E. TEXT LENGTHS ---
    df['final_text'] = df.get('extended_tweet.full_text', df.get('text', pd.Series(''))).fillna('')
    df['final_text'] = df['final_text'].where(df['final_text'] != '', df.get('text', '')).fillna('')
    
    df['text_length'] = df['final_text'].astype(str).apply(len)
    df['bio_length'] = df.get('user.description', '').astype(str).apply(len)

    # --- F. FINAL SELECTION ---
    features_to_keep = [
        # Raw user metrics
        'user.followers_count', 'user.friends_count', 'user.listed_count', 
        'user.favourites_count', 'user.statuses_count',
        # Raw tweet metrics
        'retweet_count', 'favorite_count', 'quote_count', 'reply_count',
        # Ratios & Behavioral (NEW)
        'ratio_followers_friends', 'ratio_listed_followers', 'tweets_per_day', 'account_age_days',
        'reciprocity_score', 'ratio_mention_status', 'total_tweet_engagement',
        # Booleans & Quality (UPDATED)
        'is_verified', 'is_default_profile', 'is_default_image', 'is_geo_enabled',
        'is_protected', 'has_url',
        # Time (NEW)
        'tweet_hour', 'tweet_is_weekend',
        # Length of content
        'text_length', 'bio_length', 
        # Counting of entities
        'num_urls', 'num_hashtags', 'num_mentions', 'has_media',
    ]
    
    final_cols = [c for c in features_to_keep if c in df.columns]
    
    return df[final_cols].fillna(0)



In [4]:
def create_nlp_features(df_train, df_test, y_train):
    """
    Create NLP features (TF-IDF of the bios and feeling analysis of the 
    tweets).

    Args:
        df_train (pandas.DataFrame): Full train dataframe (X_train).
        df_test (pandas.DataFrame): Test dataframe (X_kaggle).
        y_train (pandas.Series): Train target (y_train).

    Returns:
        tuple: (df_train_nlp, df_test_nlp) with the additional columns.
    """
    
    # ----------------------------------------
    # Text preprocessing
    # ----------------------------------------

    french_stopwords = stopwords.words("french")
    
    # Replace NaNs or missing values with empty strings
    train_bio = df_train.get('user.description', pd.Series([''] * len(df_train))).fillna('').astype(str)
    test_bio = df_test.get('user.description', pd.Series([''] * len(df_test))).fillna('').astype(str)

    # Retrieval of the tweet's 'final_text' (the most complete)
    # Note : We need to reproduce the 'final_text' logic of the 
    # engineering function
    def get_final_text(df):
        text = df.get('text', pd.Series([''] * len(df))).fillna('')
        full_text = df.get('extended_tweet.full_text', text).fillna(text)
        return full_text.astype(str)
        
    train_text = get_final_text(df_train)
    test_text = get_final_text(df_test)

    # ----------------------------------------
    # A. TF-IDF on the tweets (Meta-Feature)
    # ----------------------------------------
    print("TF-IDF vectorization of the tweet's body")

    # Basic preprocessing for TF-IDF
    def clean_text(text):
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # Deletion of URLs
        return text

    train_text_clean = train_text.apply(clean_text)
    test_text_clean = test_text.apply(clean_text)

    # 1. Crafting of the TF-IDF vector (learned solely on the training set)
    tfidf = TfidfVectorizer(max_features=1000, stop_words=french_stopwords, ngram_range=(2, 5), analyzer='char_wb', lowercase=False)
    X_train_tfidf_tweet = tfidf.fit_transform(train_text_clean)
    X_test_tfidf_tweet = tfidf.transform(test_text_clean)

    # 2. Training of the meta-model (logistic regression)
    log_reg = LogisticRegression(solver='sag', random_state=42)
    log_reg.fit(X_train_tfidf_tweet, y_train.astype(int))

    # 3. Extraction of the predicted probability (Meta-Feature)
    # We use the probability for class 1 (Influencer)
    train_tweet_proba = log_reg.predict_proba(X_train_tfidf_tweet)[:, 1]
    test_tweet_proba = log_reg.predict_proba(X_test_tfidf_tweet)[:, 1]    

    # ----------------------------------------
    # B. TF-IDF on the Bios (Meta-Feature)
    # ----------------------------------------
    print("  -> Computation of the TF-IDF on the Bios and training of the meta-model...")
    

    train_bio_clean = train_bio.apply(clean_text)
    test_bio_clean = test_bio.apply(clean_text)

    # 1. Computation of the TF-IDF vector (learned solely on the training set)
    tfidf = TfidfVectorizer(max_features=1000, stop_words=french_stopwords, ngram_range=(2, 5), analyzer='char_wb', lowercase=False)
    X_train_tfidf = tfidf.fit_transform(train_bio_clean)
    X_test_tfidf = tfidf.transform(test_bio_clean)

    # 2. Training of the meta-model (logistic regression)
    log_reg = LogisticRegression(solver='liblinear', random_state=42)
    log_reg.fit(X_train_tfidf, y_train.astype(int))

    # 3. Extraction of the predicted probability (Meta-Feature)
    # We use the probability for class 1 (Influencer)
    train_bio_proba = log_reg.predict_proba(X_train_tfidf)[:, 1]
    test_bio_proba = log_reg.predict_proba(X_test_tfidf)[:, 1]

    # ----------------------------------------
    # B. Sentiment analysis (Polarity and Subjectivity)
    # ----------------------------------------
    print("  -> Extraction of the Sentiment (Polarity/Subjectivity) of the Tweets...")
    
    # The TextBlob function is used to get the sentiment scores
    # This is a slow operation, we just need to be patient
    def get_sentiment(text):
        try:
            analysis = TextBlob(text)
            return pd.Series({'polarity': analysis.sentiment.polarity, 'subjectivity': analysis.sentiment.subjectivity})
        except:
            return pd.Series({'polarity': 0.0, 'subjectivity': 0.0})

    train_sentiment = train_text.apply(get_sentiment)
    test_sentiment = test_text.apply(get_sentiment)

    # ----------------------------------------
    # 4. Fusion of the NLP features
    # ----------------------------------------
    
    # Creation of the NLP features dataframes
    df_train_nlp = pd.DataFrame({
        'meta_bio_proba': train_bio_proba,
        'tweet_polarity': train_sentiment['polarity'],
        'tweet_subjectivity': train_sentiment['subjectivity'],
        'meta_tweet_proba': train_tweet_proba
    })
    
    df_test_nlp = pd.DataFrame({
        'meta_bio_proba': test_bio_proba,
        'tweet_polarity': test_sentiment['polarity'],
        'tweet_subjectivity': test_sentiment['subjectivity'],
        'meta_tweet_proba': test_tweet_proba
    })

    return df_train_nlp, df_test_nlp

### Computation of the datasets

In [5]:
# =======================================================
# We compute the actual datasets for each part of the full model
# =======================================================
print("🛠️ Construction des features avancées (Métadonnées)...")

# Application of the robust function (metadata)
X_train_metadata = create_advanced_features(X_train)
X_kaggle_metadata = create_advanced_features(X_kaggle)

# Preparation of the target
y_train_clean = y_train.astype(int)

# --- NEW STEP : CREATION OF THE NLP FEATURES ---
X_train_nlp, X_kaggle_nlp = create_nlp_features(X_train, X_kaggle, y_train_clean)

🛠️ Construction des features avancées (Métadonnées)...


/tmp/ipykernel_9914/2729300661.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['user_created_at_dt'] = pd.to_datetime(df.get('user.created_at'), errors='coerce')
/tmp/ipykernel_9914/2729300661.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['user_created_at_dt'] = pd.to_datetime(df.get('user.created_at'), errors='coerce')


TF-IDF vectorization of the tweet's body


/home/aprats/Documents/Polytechnique/3A/P1/CSC_51054_EP/InTweetionists/.venv/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:539: UserWarning: The parameter 'stop_words' will not be used since 'analyzer' != 'word'
  warnings.warn(


  -> Computation of the TF-IDF on the Bios and training of the meta-model...


/home/aprats/Documents/Polytechnique/3A/P1/CSC_51054_EP/InTweetionists/.venv/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:539: UserWarning: The parameter 'stop_words' will not be used since 'analyzer' != 'word'
  warnings.warn(


  -> Extraction of the Sentiment (Polarity/Subjectivity) of the Tweets...


### Definition of functions for Word2Vec

In [6]:
def simple_tokenize(text):
    """
    Tokenise un texte en le mettant en minuscules et en le séparant par espace.
    Nettoie les caractères non alphanumériques courants pour Word2Vec.
    """
    # 1. Switch to lower case
    text = text.lower()
    
    # 2. Replacement of the punctuation with spaces for separation
    # This makes things easier for W2V
    text = re.sub(r'[.,;:`"\'!?()]', ' ', text)
    
    # 3. Séparation par espace et suppression des entrées vides
    return [word for word in text.split() if word]



### LOADING AND PREPROCESSING OF THE DATA

In [7]:

# --- 1. TEXT EXTRACTION FUNCTION ---
def extract_full_text(row):
    """Extract the most complete text available from a given tweet.
    
    Args:
        row : A single recording in the dataframe.
    
    Returns:
        The full text if available, if not then the standard text.
    """
    if "extended_tweet.full_text" in row and pd.notna(row["extended_tweet.full_text"]):
        return row["extended_tweet.full_text"]
    if "text" in row and pd.notna(row["text"]):
        return row["text"]
    return ""

# --- 2. LOADING OF THE RAW DATA ---
try:
    # Application of the text extraction
    train_data["full_text"] = train_data.apply(extract_full_text, axis=1)
    kaggle_data["full_text"] = kaggle_data.apply(extract_full_text, axis=1)

    # DEFINTION OF THE VARIABLES FOR THE W2V PIPELINE
    X_full = train_data["full_text"].values # Full train text
    y_full = train_data["label"].values.astype(int) # Labels
    X_kaggle_full = kaggle_data["full_text"].values # Full kaggle text
    
    print("Raw data (X_full, y_full) loaded and ready. ✓")

except FileNotFoundError as e:
    print(f"❌ ERROR: File not found. Check the filepath : {e}")
    exit()

Raw data (X_full, y_full) loaded and ready. ✓


### Word2Vec configuration

In [8]:
EMBEDDING_DIM = 250 
WINDOW_SIZE = 5
MIN_COUNT = 1

### Preprocessing of the text data (Tokenization)

In [9]:
print("\n🔄 Tokenization of the text data...")
# 1. Tokenization of the train set (X_full)
X_full_tokenized = [simple_tokenize(text) for text in X_full]

# 2. Tokenization of the Kaggle set (X_kaggle_full)
X_kaggle_tokenized = [simple_tokenize(text) for text in X_kaggle_full]


🔄 Tokenization of the text data...


### Vectorization function

In [10]:
def document_vectorizer(tokens, model, dim):
    """Compute the mean vector for a document."""
    vector = np.zeros(dim)
    count = 0
    # Only comutes the mean over the words belonging to the W2V 
    # vocabulary
    for word in tokens:
        if word in model.wv:
            vector += model.wv[word]
            count += 1
    
    if count != 0:
        vector /= count
        
    return vector

### Generation of the embeddings (vectors)

In [11]:
print("🔄 Génération of the W2V embeddings by mean...")

w2v_model = Word2Vec(
    sentences=X_full_tokenized, 
    vector_size=EMBEDDING_DIM, 
    window=WINDOW_SIZE, 
    min_count=MIN_COUNT, 
    sg=1 # 1: Skip-gram (often better than C-bow for W2V)
)

# Training
X_train_vectors = np.array([document_vectorizer(tokens, w2v_model, EMBEDDING_DIM) for tokens in X_full_tokenized])

# Kaggle
X_kaggle_vectors = np.array([document_vectorizer(tokens, w2v_model, EMBEDDING_DIM) for tokens in X_kaggle_tokenized])

# Creation of the embedding dataframes
embed_cols = [f'w2v_e_{i}' for i in range(EMBEDDING_DIM)]

X_train_nlp = pd.DataFrame(X_train_vectors, columns=embed_cols)
X_kaggle_nlp = pd.DataFrame(X_kaggle_vectors, columns=embed_cols)

🔄 Génération of the W2V embeddings by mean...


### Creation of the tensors to feed the model

In [12]:
# Features and labels
X_train_metadata_tensor = torch.tensor(X_train_metadata.values, dtype=torch.float32)
X_train_nlp_tensor = torch.tensor(X_train_nlp.values, dtype=torch.float32)

X_kaggle_metadata_tensor = torch.tensor(X_kaggle_metadata.values, dtype=torch.float32)
X_kaggle_nlp_tensor = torch.tensor(X_kaggle_nlp.values, dtype=torch.float32)


y_train_tensor = torch.tensor(y_train_clean.values, dtype=torch.long)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X_train_metadata_tensor = X_train_metadata_tensor.to(device)
X_train_nlp_tensor = X_train_nlp_tensor.to(device)
X_kaggle_metadata_tensor = X_kaggle_metadata_tensor.to(device)
X_kaggle_nlp_tensor = X_kaggle_nlp_tensor.to(device)
y_train_tensor = y_train_tensor.to(device)


### Definition of the architecture combining shallow and deep learning models

In [13]:
class TabTransformerLite(nn.Module):
    def __init__(self, input_dim, emb_dim=32, num_heads=4, num_layers=3, dropout=0.1):
        super().__init__()

        self.feature_embedding = nn.Linear(1, emb_dim)

        # Normalization after the embedding
        self.pre_norm = nn.LayerNorm(emb_dim)

        # Very light Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=num_heads,
            dim_feedforward=emb_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x):
        x = x.unsqueeze(-1)
        x = self.feature_embedding(x)
        x = self.pre_norm(x)
        x = self.transformer(x)
        x = x.mean(dim=1)
        return x


class MetaEncoder(nn.Module):
    def __init__(self, input_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)


class FusionClassifier(nn.Module):
    def __init__(self, text_dim, meta_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(text_dim + meta_dim, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.net(x)


class UnifiedModel(nn.Module):
    def __init__(self, vocab_size, meta_dim, num_classes,
                 text_emb_dim=128, text_heads=4, text_layers=2):
        super().__init__()

        self.text_encoder = TabTransformerLite(
            input_dim=vocab_size,
            emb_dim=text_emb_dim,
            num_heads=text_heads,
            num_layers=text_layers
        )

        self.meta_encoder = MetaEncoder(
            input_dim=meta_dim,
            hidden=128
        )

        self.classifier = FusionClassifier(
            text_dim=text_emb_dim,
            meta_dim=128,
            num_classes=num_classes
        )

    def forward(self, tokens, metadata):
        h_text = self.text_encoder(tokens)
        h_meta = self.meta_encoder(metadata)

        h = torch.cat([h_text, h_meta], dim=1)

        return self.classifier(h)


### Instantiation

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UnifiedModel(
    vocab_size=X_train_nlp.shape[1],
    meta_dim=X_train_metadata.shape[1],
    num_classes=2,
    text_emb_dim=256,
    text_heads=8,
    text_layers=15
).to(device)


In [15]:
train_dataset_metadata = TensorDataset(X_train_metadata_tensor, y_train_tensor)
train_dataset_nlp = TensorDataset(X_train_nlp_tensor, y_train_tensor)
loader_metadata = DataLoader(train_dataset_metadata, batch_size=64, shuffle=True)
loader_nlp = DataLoader(train_dataset_nlp, batch_size=64, shuffle=True)
loader_metadata.__dict__

{'dataset': <torch.utils.data.dataset.TensorDataset at 0x7f834b00ea50>,
 'num_workers': 0,
 'prefetch_factor': None,
 'pin_memory': False,
 'pin_memory_device': '',
 'timeout': 0,
 'worker_init_fn': None,
 '_DataLoader__multiprocessing_context': None,
 'in_order': True,
 '_dataset_kind': 0,
 'batch_size': 64,
 'drop_last': False,
 'sampler': <torch.utils.data.sampler.RandomSampler at 0x7f834b00e7b0>,
 'batch_sampler': <torch.utils.data.sampler.BatchSampler at 0x7f834b00e660>,
 'generator': None,
 'collate_fn': <function torch.utils.data._utils.collate.default_collate(batch)>,
 'persistent_workers': False,
 '_DataLoader__initialized': True,
 '_IterableDataset_len_called': None,
 '_iterator': None}

### Model training

In [16]:
def predict_in_batches(model, X_nlp, X_metadata, batch_size=256):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X_nlp), batch_size):
            xbnlp = X_nlp[i:i+batch_size].to(device)
            xbmeta = X_metadata[i:i+batch_size].to(device)
            pb = model(xbnlp, xbmeta).cpu()
            preds.append(pb)
    return torch.cat(preds, dim=0)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
epochs = 50

patience = 5
patience_counter = 0
best_acc = 0.0
best_state = None

for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    for (xbnlp, yb), (xbmeta, yb) in zip(loader_nlp, loader_metadata):
        optimizer.zero_grad()
        out = model(xbnlp, xbmeta)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # Accuracy train
    model.eval()
    with torch.no_grad():
        preds = predict_in_batches(model, X_train_nlp_tensor, X_train_metadata_tensor)
        preds = preds.to(device)
        acc = (preds.argmax(1) == y_train_tensor).float().mean().item()

    print(f"[Epoch {epoch+1}] Loss={total_loss/len(loader_nlp):.4f} | Acc={acc:.4f}")

    # Early stopping
    if acc > best_acc:
        best_acc = acc
        best_state = model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print("Early stopping triggered ⛔")
        break

model.load_state_dict(best_state)


### Save of the model after the training

In [ ]:
filename = f"model_weights_{datetime.now()}.pth"
filename = filename.replace(" ", "_")
filename = filename.replace(":", "-")
filename = filename.replace(".", "-")
torch.save(model.state_dict(), filename)

### Computation of the dataset to be submitted on kaggle

In [17]:
model.load_state_dict(torch.load("model_weights_2025-12-07_02-53-18-274038.pth", weights_only=True))

<All keys matched successfully>

In [21]:
# The dataset to label is too heavy for the GPU so we load everything 
# back to the RAM
model_cpu = model.to("cpu")

batch_size = 2048
preds = []
probas = []

model_cpu.eval()
with torch.no_grad():
    for i in range(0, len(X_kaggle_metadata_tensor), batch_size):
        batch_nlp = X_kaggle_nlp_tensor[i:i+batch_size].to("cpu")
        batch_metadata = X_kaggle_metadata_tensor[i:i+batch_size].to("cpu")
        out = model_cpu(batch_nlp, batch_metadata)
        preds.append(out.argmax(1).numpy())
        probas.append(out.numpy())

y_pred_proba_kaggle = np.concatenate(probas)
y_pred_kaggle = np.concatenate(preds)

USER_ID_COLUMN = 'user.profile_banner_url'

y_pred_kaggle = (y_pred_kaggle > 0.5).astype(int)



In [22]:


df_kaggle_preds = pd.DataFrame({
    'challenge_id': X_kaggle['challenge_id'],
    'user_id_key': X_kaggle[USER_ID_COLUMN], # User key
    'y_pred_proba_observer': y_pred_proba_kaggle[:, 0],
    'y_pred_proba_influencer': y_pred_proba_kaggle[:, 1],           
    'y_pred_tweet': y_pred_kaggle
})



In [23]:
y_pred_proba_kaggle

array([[-0.18830094,  0.60280776],
       [-0.2866996 ,  3.156818  ],
       [-0.2543932 , -1.1321622 ],
       ...,
       [-0.15513846,  1.4704261 ],
       [-0.07826161,  0.48499447],
       [ 0.10298989, -2.7662542 ]], dtype=float32)

In [25]:
df_kaggle_preds.to_csv('AVEC_PROBAS_SUPER_IMPORTANT.csv', index=False)

#### Grouping all the tweets by user

In [ ]:

user_pred_mean = df_kaggle_preds.groupby('user_id_key')['y_pred_tweet'].mean()
user_majority_vote = np.where(user_pred_mean >= 0.5, 1, 0)

# We convert the majority vote series to a dataframe to prepare for 
# fusion
df_majority_vote = pd.DataFrame({
    'user_id_key': user_pred_mean.index,
    'y_pred_user_majority': user_majority_vote
})

df_final_preds = pd.merge(
    df_kaggle_preds,
    df_majority_vote,
    on='user_id_key',
    how='left'
)

df_final_preds['y_pred_final'] = df_final_preds['y_pred_user_majority'].fillna(
    df_final_preds['y_pred_tweet']
)
y_final_submission = df_final_preds['y_pred_final']
output = pd.DataFrame({
    'ID': df_final_preds['challenge_id'],
    "Prediction": y_final_submission
})

output['Prediction'] = output['Prediction'].astype(int)
output.to_csv('submission_lightgbm_majority_vote_final_nan_fixed.csv', index=False)

### Save submission

In [ ]:
output_dl = pd.DataFrame({
    'ID': X_kaggle['challenge_id'],
    'Prediction': y_pred_kaggle
})

In [ ]:
output_dl.to_csv('submission_unified.csv', index=False)
print("✓ Submission for PyTorch model saved as 'submission_unified.csv'")

### Useful plots : model architecture and data distribution

In [ ]:
print(model)

In [ ]:
import matplotlib.pyplot as plt
# draw a histogram of the proba column
user_pred_mean.hist(bins=100)

# add labels and title
plt.xlabel('Proba')
plt.ylabel('Frequency')
plt.title('Distribution of proba')


We notice that the data is well separated so there is not much room for improvement based on setting a different separation threshold